# Khám phá LEMMA-RCA Dataset

> **LEMMA-RCA: A Large Multi-modal Multi-domain Dataset for Root Cause Analysis** (NEC Labs & Collaborators)
>
> Mục tiêu: Đọc hiểu và phân tích cấu trúc log, metrics, trace Jaeger và Ground Truth tiêm lỗi (DDoS OOM, Resource Contention, Storage Failure, Noisy Neighbor) trong hệ thống microservice **Book-Info** chạy trên nền tảng **OpenShift**.

## 1. Thiết lập môi trường và nạp module Clean Architecture

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Thêm root repo (chứa package src) vào sys.path
notebook_dir = Path.cwd().resolve()
explore_root = notebook_dir.parents[3]
if str(explore_root) not in sys.path:
    sys.path.insert(0, str(explore_root))

from src.datasets.lemma_rca.application.use_cases import (
    describe_lemma_use_case,
    fetch_lemma_use_case,
    list_lemma_incidents_use_case,
    get_lemma_incident_use_case,
    inspect_lemma_logs_use_case,
)
from src.datasets.lemma_rca.infrastructure.loader import LemmaLoader
print("Module LEMMA-RCA đã sẵn sàng!")

## 2. Kế hoạch tải dữ liệu (Dry-run & Sample Fetch)

Dataset LEMMA-RCA có tổng dung lượng full lên tới ~51GB (4 file ZIP lớn). Bằng cơ chế HTTP Range thông qua `fsspec`, ta có thể xem trước kế hoạch (dry-run) và chỉ tải các file log mẫu (`sample` mode, ~47MB) với 20,000+ dòng log và slide Ground Truth.

In [ ]:
# Kiểm tra kế hoạch tải (Dry-run)
plan = fetch_lemma_use_case(mode="sample", dry_run=True)
print(f"Mode: {plan.mode}")
print(f"Tổng dung lượng dự kiến: {plan.total_estimated_bytes / (1024*1024):.2f} MB")
for item in plan.items:
    print(f" - {item.name}: {item.estimated_size_bytes / (1024*1024):.2f} MB")

## 3. Khảo sát 4 tiêu chuẩn bắt buộc (Shape, Columns, Head, Value Counts)

In [ ]:
profile = describe_lemma_use_case(sample_limit=10000)
print(f"Tên dataset: {profile['name']}")
print(f"Kích thước (Shape): ({profile['n_rows']}, {profile['n_cols']})")
print(f"Cột dữ liệu: {profile['columns']}")

print("\n--- 10 DÒNG ĐẦU TIÊN (HEAD) ---")
df_head = pd.DataFrame(profile['head'])
display(df_head[['timestamp', 'service', 'level', 'message', 'pod']])

print("\n--- TẦN SUẤT PHÂN BỐ (VALUE COUNTS) ---")
for col, vc in profile['value_counts'].items():
    print(f"\nCột [{col}]:")
    for val, cnt in vc.items():
        print(f"  {val:<20}: {cnt:>6}")

## 4. Xem danh sách Ground Truth và Kịch bản Sự cố (Incidents)

Trích xuất từ slide tài liệu chính thức (`20220606/README.pptx`), kết quả JMeter và bài báo công bố.

In [ ]:
incidents = list_lemma_incidents_use_case()
inc_table = []
for inc in incidents:
    inc_table.append({
        "ID": inc.incident_id,
        "Hệ thống": inc.system,
        "Ngày": inc.date,
        "Loại lỗi": inc.fault_type,
        "Root Cause": inc.root_cause_service
    })
display(pd.DataFrame(inc_table))

# Xem chi tiết sự cố DDoS OOM
ddos_inc = get_lemma_incident_use_case("lemma-20220606-ddos-oom")
print("\nChi tiết sự cố DDoS OOM:")
print(f" - Mô tả: {ddos_inc.description}")
print(f" - Triệu chứng: {ddos_inc.symptoms}")
print(f" - Dịch vụ bị ảnh hưởng: {ddos_inc.impacted_services}")

## 5. Phân tích nội dung Log chi tiết (Lọc ERROR / WARN và từ khóa)

In [ ]:
# Lọc các log có level ERROR
errors = inspect_lemma_logs_use_case(level="ERROR", limit=10)
print(f"Tìm thấy {len(errors)} dòng log ERROR:")
for e in errors:
    print(f"[{e.timestamp_str}] [{e.service}] Pod: {e.pod_name}")
    print(f"  -> Message: {e.message}\n")

# Lọc log liên quan đến service productpage
pp_logs = inspect_lemma_logs_use_case(service="productpage", limit=5)
print("\nLog mẫu của service productpage:")
for p in pp_logs:
    print(f"[{p.timestamp_str}] [{p.level}] {p.message}")

## 6. Chuyển đổi thành Pandas DataFrame để tiếp tục khai thác

In [ ]:
loader = LemmaLoader()
canonical_rows = loader.load_canonical_rows(limit=5000)
df = pd.DataFrame(canonical_rows)
print("DataFrame shape:", df.shape)
print(df.info())
display(df.sample(5))